In [1]:
import pandas as pd
import numpy as np
from datetime import timedelta,datetime
import os

In [2]:
df = pd.read_csv('processed_dataset.csv')
df

,e_datetime,did,func,d_value
0,2010-11-04 00:03:50.209589,M003,motion,ON
1,2010-11-04 00:03:57.399391,M003,motion,OFF
2,2010-11-04 00:15:08.984841,T002,temperature,21.5
3,2010-11-04 00:30:19.185547,T003,temperature,21
4,2010-11-04 00:30:19.385336,T004,temperature,21
...,...,...,...,...
1719547,2011-06-11 23:42:59.28507,T002,temperature,25.5
1719548,2011-06-11 23:48:02.888409,T001,temperature,23.5
1719549,2011-06-11 23:48:02.988798,T002,temperature,25
1719550,2011-06-11 23:53:06.4292,T002,temperature,25.5


In [7]:
# 将d_value转换为数值
df['d_value'] = df['d_value'].replace({"ON": 1.0, "OFF": 0.0, "OPEN": 1.0, "CLOSE": 0.0})
# 将不带毫秒的e_datetime增加毫秒
df.loc[df['e_datetime'].str.contains(r'^[^.]*$'), 'e_datetime']+= '.0'

In [8]:
# 搜索d_value异常数据
character_values = df[df['d_value'].apply(lambda x: isinstance(x, str))]['d_value']

# 去除重复值，得到唯一的字符数据列表
my_list = list(character_values.unique())
f_list = my_list.copy()
print(f_list)
for value in my_list:
    try:
        if float(value)<=50:
            f_list.pop(f_list.index(value))
    except:
        pass

error_value = pd.DataFrame()
for charv in f_list:
    error_value = pd.concat([error_value, df[df['d_value'] == charv]])
error_value

['21.5', '21', '20.5', '20', '19.5', '19', '18.5', '26.5', '23', '22.5', '23.5', '22', '24.5', '25', '28', '30.5', '27', '26', '32', '28.5', '27.5', '24', '33', '29.5', '29', '34', '30', '34.5', '31.5', '25.5', '35', '36', '36.5', '32.5', '37', '33.5', '37.5', '31', '35.5', '38', '38.5', '39', '18', '17.5', '17', '39.5', '16.5', '16', 'ONc', 'OFFc', 'OFF5', 'OFcF', 'ON5', 'ON55', 'OFFcc', 'OFF5cc', 'OFF5c', 'ON5c', '28.55c', 'OFFc5', 'ONcc', 'ONc5c', '26cc', '285', 'ONc5', 'OPENc', 'OcFF', 'OFFccc5', '19.55', '245', '225', '23.555', '235', '42', 'O', 'OF', 'ONM026', '41.5', '42.5', '43', '40.5', 'ONM009', 'ONM024', 'CLOSED']


,e_datetime,did,func,d_value
275238,2010-12-13 09:42:01.868596,M020,motion,ONc
275270,2010-12-13 09:44:09.502301,M019,motion,ONc
275336,2010-12-13 09:50:05.056768,M012,motion,ONc
275385,2010-12-13 09:57:25.888599,M021,motion,ONc
275468,2010-12-13 10:00:51.043461,M015,motion,ONc
...,...,...,...,...
1268321,2011-04-14 09:56:45.413465,M026,motion,ONM026
1268560,2011-04-14 10:19:01.978394,M020,motion,ONM026
1355844,2011-04-25 00:15:41.512656,M010,motion,ONM009
1504084,2011-05-14 15:54:34.668259,M024,motion,ONM024


In [ ]:
# 清洗异常数据
df = df.drop(error_value.index)

In [ ]:
#将d_value列转化为float
df['d_value'] = df['d_value'].astype(float)
df

In [ ]:
def round_timestamp(s):
    s = s.apply(lambda x: datetime.strptime(x, '%Y-%m-%d %H:%M:%S.%f'))
    seconds = s.dt.microsecond  # 获取小数部分（秒）
    max_digits = seconds.map(lambda x: len(str(x))).max()
    seconds0 = seconds.map(lambda x: format(x / (10 ** max_digits), f'.{max_digits}f'))
    rounded_seconds = round(seconds0.astype(float))  # 四舍五入修约
    s = s.apply(lambda x: x.replace(microsecond=0))  # 将微秒部分置零
    s += pd.to_timedelta(rounded_seconds, unit='s')  # 加上修约后的秒数
    # 处理分界线
    s = s.apply(lambda x: x.replace(year=2077))  # 将年份替换为2077
    s = s.apply(lambda x: x.timestamp())  # 转换为timestamp数字
    return s

# 数据集切割
def chunk(chunk_size, data):
    chunks = np.split(data, range(chunk_size, len(data), chunk_size))
    return chunks
# 数据集合并
def merge_dataset(chunk_size, e_datetime, encoded_did, encoded_func, d_value):
    v1 = chunk(chunk_size, e_datetime)
    v2 = chunk(chunk_size, encoded_did)
    v3 = chunk(chunk_size, encoded_func)
    v4 = chunk(chunk_size, d_value)
    print(len(v1))
    for i in range(len(v1)):
        print(f'第{i}个数据集')
        print(v1[i].shape,v2[i].shape,v3[i].shape,v4[i].shape)
        merged_array = np.concatenate((v1[i].reshape(-1, 1), v2[i], v3[i], v4[i].reshape(-1, 1)), axis=1)
        np.save(f'{file_dir}/dataset{i}.npy', merged_array)

In [ ]:
df['e_datetime'] = round_timestamp(df['e_datetime'].copy())
e_max = df['e_datetime'].max()
e_min = df['e_datetime'].min()
df

In [ ]:
from sklearn.preprocessing import MinMaxScaler

In [ ]:
# 将某一列归一化
def normalize_column(df, column_name):
    # 创建一个MinMaxScaler对象
    scaler = MinMaxScaler()
    column_to_normalize = df[column_name].copy()
    
    # 从DataFrame中提取指定列，并进行reshape
    column_to_normalize = column_to_normalize.values.reshape(-1, 1)

    # 对列进行归一化
    normalized_column = scaler.fit_transform(column_to_normalize)

    # 将归一化后的数据替换原始列
    return normalized_column

def denormalize_column(normalized_data, original_min, original_max):
    # 创建一个MinMaxScaler对象
    scaler = MinMaxScaler()

    # 对scaler对象进行拟合
    scaler.fit(np.array([[original_min], [original_max]]).reshape(-1, 1))

    # 根据输入数据的类型决定是否进行reshape
    if isinstance(normalized_data, (int, float)):
        normalized_data = np.array([[normalized_data]])
    else:
        normalized_data = np.array(normalized_data).reshape(-1, 1)

    # 将归一化后的数据进行逆转换
    denormalized_data = scaler.inverse_transform(normalized_data)

    return denormalized_data.flatten()

In [ ]:
df['e_datetime'] = normalize_column(df, 'e_datetime')
df

In [ ]:
# de_df = denormalize_column(df['e_datetime'],e_min,e_max)
# de_num = denormalize_column(df.iloc[0,0],e_min,e_max)

In [ ]:
df

In [ ]:
from sentence_transformers import SentenceTransformer

In [ ]:
import torch
# 检查是否可用CUDA
if torch.cuda.is_available():
    print('cuda')
    # 指定CUDA设备
    device = torch.device('cuda')
else:
    print('cpu')
    device = torch.device('cpu')

# 加载SentenceTransformer模型并将其移动到指定的设备上
model = SentenceTransformer('distilbert-base-nli-mean-tokens').to(device)

In [ ]:
file_dir = 'E:/dataset/aruba1'
dataset_num = 0
if os.path.exists(file_dir):
    print('Y')
    dataset = np.load(f'{file_dir}/dataset{dataset_num}.npy')
    print(dataset.shape)
else:
    print('N')
    os.mkdir(file_dir)
    #编码
    encoded_did = model.encode(df['did'].values.tolist())
    encoded_func = model.encode(df['func'].values.tolist())
    e_datetime = df['e_datetime'].to_numpy()
    d_value = df['d_value'].to_numpy()
    merge_dataset(200000, e_datetime, encoded_did, encoded_func, d_value)